In [3]:
# Install packages and remove torchao
!pip uninstall -y torchao
!pip install -q -U transformers accelerate datasets sentencepiece

In [4]:
# Imports and configuration
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed,
)
import warnings
warnings.filterwarnings("ignore")

SEED = 42
set_seed(SEED)

TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

DEBERTA_NAME = 'microsoft/deberta-v3-small'
ROBERTA_NAME = 'FacebookAI/roberta-base'

DEBERTA_OUT = '/kaggle/working/deberta-v3-small-finetuned'
ROBERTA_OUT = '/kaggle/working/roberta-base-finetuned'

MAX_LENGTH = 384
LABEL2ID = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
OPTIONS = ['A', 'B', 'C', 'D', 'E']

W_DEB = 0.70
W_ROB = 0.30
INSTR_PREFIX = 'Answer the following multiple-choice question carefully:\n\n'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

In [5]:
# Load data and build prompt text
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

def create_prompt(df):
    return (
        'Question: ' + df['prompt'].fillna('') + '\n\n'
        'Option A: ' + df['A'].fillna('') + '\n'
        'Option B: ' + df['B'].fillna('') + '\n'
        'Option C: ' + df['C'].fillna('') + '\n'
        'Option D: ' + df['D'].fillna('') + '\n'
        'Option E: ' + df['E'].fillna('')
    )

train_df['text'] = create_prompt(train_df)
test_df['text'] = create_prompt(test_df)
train_df['label'] = train_df['answer'].map(LABEL2ID)
print(f'train: {len(train_df)} rows, test: {len(test_df)} rows')

train: 2000 rows, test: 500 rows


In [6]:
# Stratified train validation split
train_split, valid_split = train_test_split(
    train_df, test_size=0.1, stratify=train_df['label'], random_state=SEED
)
train_split = train_split.reset_index(drop=True)
valid_split = valid_split.reset_index(drop=True)
print(f'train_split: {len(train_split)}, valid_split: {len(valid_split)}')

train_split: 1800, valid_split: 200


In [7]:
# MAP@3 metric and compute metrics function
def map3_score(logits, labels):
    top3 = np.argsort(logits, axis=1)[:, -3:][:, ::-1]
    score = 0.0
    for pred, true in zip(top3, labels):
        if true in pred:
            score += 1.0 / (int(np.where(pred == true)[0][0]) + 1)
    return score / len(labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'map3': map3_score(logits, labels),
    }

In [8]:
# Fine-tune helper
def finetune_model(base_name, out_dir):
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(base_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_name, num_labels=5, id2label=ID2LABEL, label2id=LABEL2ID,
        torch_dtype=torch.float32,
    )

    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

    trn_ds = Dataset.from_pandas(train_split[['text', 'label']]).map(
        tokenize, batched=True
    )
    val_ds = Dataset.from_pandas(valid_split[['text', 'label']]).map(
        tokenize, batched=True
    )
    keep = ['input_ids', 'attention_mask', 'label']
    trn_ds = trn_ds.remove_columns([c for c in trn_ds.column_names if c not in keep])
    val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep])

    args = TrainingArguments(
        output_dir=out_dir + '-runs',
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='steps',
        logging_steps=50,
        learning_rate=1e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        weight_decay=0.01,
        warmup_steps=100,
        max_grad_norm=1.0,
        fp16=False,
        bf16=use_bf16,
        load_best_model_at_end=True,
        metric_for_best_model='map3',
        greater_is_better=True,
        report_to='none',
        save_total_limit=1,
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=trn_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    print(f'final eval: {trainer.evaluate()}')
    del trainer, model
    torch.cuda.empty_cache()

In [9]:
# Fine-tune DeBERTa
finetune_model(DEBERTA_NAME, DEBERTA_OUT)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Map3
1.063721,0.270546,5,0.985000,0.985255,0.985000,0.984952,0.990000


final eval: {'eval_loss': 0.27054640650749207, 'eval_accuracy': 0.985, 'eval_precision': 0.9852553763440861, 'eval_recall': 0.985, 'eval_f1': 0.9849521240920701, 'eval_map3': 0.99}


In [10]:
# Fine-tune RoBERTa
finetune_model(ROBERTA_NAME, ROBERTA_OUT)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Map3
0.444274,0.112611,5,1.000000,1.000000,1.000000,1.000000,1.000000


final eval: {'eval_loss': 0.11261129379272461, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0, 'eval_map3': 1.0}


In [11]:
# Load fine-tuned models and helpers
deb_tok = AutoTokenizer.from_pretrained(DEBERTA_OUT)
deb_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_OUT).to(device).eval()

rob_tok = AutoTokenizer.from_pretrained(ROBERTA_OUT)
rob_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_OUT).to(device).eval()

@torch.no_grad()
def predict_probs_batch(model, tokenizer, texts, batch_size=8):
    probs_list = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors='pt').to(device)
        logits = model(**enc).logits.float()
        probs_list.append(F.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(probs_list, axis=0)

def top3_string(probs_row):
    return ' '.join(OPTIONS[i] for i in np.argsort(-probs_row)[:3])

def top3_letters(probs_row):
    return [OPTIONS[i] for i in np.argsort(-probs_row)[:3]]

print('models loaded')

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

models loaded


In [12]:
# Precompute predictions for test and validation
test_texts = test_df['text'].tolist()
deb_test_probs = predict_probs_batch(deb_model, deb_tok, test_texts)
rob_test_probs = predict_probs_batch(rob_model, rob_tok, test_texts)
wens_test_probs = W_DEB * deb_test_probs + W_ROB * rob_test_probs

aug_texts_50 = [INSTR_PREFIX + t for t in test_df['text'].iloc[:50].tolist()]
deb_aug_probs_50 = predict_probs_batch(deb_model, deb_tok, aug_texts_50)

valid_texts_100 = valid_split['text'].iloc[:100].tolist()
valid_labels_100 = valid_split['label'].iloc[:100].tolist()
deb_valid_probs = predict_probs_batch(deb_model, deb_tok, valid_texts_100)
rob_valid_probs = predict_probs_batch(rob_model, rob_tok, valid_texts_100)
wens_valid_probs = W_DEB * deb_valid_probs + W_ROB * rob_valid_probs

print(f'test probs: {deb_test_probs.shape}, valid probs: {deb_valid_probs.shape}')

test probs: (500, 5), valid probs: (100, 5)


In [23]:
# Q1 DeBERTa top prediction row 25
row25_text = train_df.iloc[25]['text']
p_deb_25 = predict_probs_batch(deb_model, deb_tok, [row25_text])[0]
p_rob_25 = predict_probs_batch(rob_model, rob_tok, [row25_text])[0]
top1_deb_25 = int(np.argmax(p_deb_25))
print(f'Q1: {OPTIONS[top1_deb_25]}, {p_deb_25[top1_deb_25]:.4f}')
print(f'    full probs (A,B,C,D,E): {p_deb_25}')

Q1: E, 0.7676
    full probs (A,B,C,D,E): [0.04839091 0.01088661 0.13473581 0.03838363 0.76760304]


In [24]:
# Q2 simple average ensemble row 25
p_avg_25 = (p_deb_25 + p_rob_25) / 2
top1_avg_25 = int(np.argmax(p_avg_25))
print(f'Q2: {OPTIONS[top1_avg_25]}')
print(f'    avg probs: {p_avg_25}')

Q2: E
    avg probs: [0.03264464 0.01759848 0.07632109 0.02177887 0.8516569 ]


In [25]:
# Q3 weighted ensemble row 25
p_wens_25 = W_DEB * p_deb_25 + W_ROB * p_rob_25
top1_wens_25 = int(np.argmax(p_wens_25))
print(f'Q3: {OPTIONS[top1_wens_25]}')
print(f'    weighted probs: {p_wens_25}')

Q3: E
    weighted probs: [0.03894314 0.01491373 0.09968698 0.02842077 0.81803536]


In [26]:
# Q4 top-3 string row 25
q4 = top3_string(p_wens_25)
print(f'Q4: {q4}')

Q4: E C A


In [27]:
# Q5 full submission file
preds = [top3_string(wens_test_probs[i]) for i in range(len(test_df))]
sub = pd.DataFrame({'ID': test_df['id'], 'Prediction': preds})
sub.to_csv('submission.csv', index=False)
q5 = len(sub)
print(f'Q5: {q5}')
print(sub.head())

Q5: 500
   ID Prediction
0   1      A D E
1   2      B D C
2   3      B D C
3   4      E C D
4   5      C A B


In [28]:
# Q6 TTA top-1 changes on first 50
deb_test_50 = deb_test_probs[:50]
tta_avg_50 = (deb_test_50 + deb_aug_probs_50) / 2
orig_top1 = np.argmax(deb_test_50, axis=1)
tta_top1 = np.argmax(tta_avg_50, axis=1)
q6 = int((orig_top1 != tta_top1).sum())
print(f'Q6: {q6}')

Q6: 0


In [29]:
# Q7 DeBERTa vs ensemble disagreement 100 rows
deb100 = deb_test_probs[:100]
wens100 = wens_test_probs[:100]
deb_top1 = np.argmax(deb100, axis=1)
wens_top1 = np.argmax(wens100, axis=1)
q7 = int((deb_top1 != wens_top1).sum())
print(f'Q7: {q7}')

Q7: 0


In [30]:
# Q8 positive confidence gain 100 rows
conf_deb = deb100.max(axis=1)
conf_wens = wens100.max(axis=1)
q8 = int(((conf_wens - conf_deb) > 0).sum())
print(f'Q8: {q8}')

Q8: 94


In [31]:
# Q9 top-3 ordering changes 100 rows
q9 = 0
for i in range(100):
    if top3_string(deb100[i]) != top3_string(wens100[i]):
        q9 += 1
print(f'Q9: {q9}')

Q9: 15


In [32]:
# Q10 MAP@3 on validation ensemble
def map3_row(pred_letters, correct_letter):
    return 1.0 / (pred_letters.index(correct_letter) + 1) if correct_letter in pred_letters else 0.0

scores = []
for i in range(100):
    top3 = top3_letters(wens_valid_probs[i])
    correct = ID2LABEL[int(valid_labels_100[i])]
    scores.append(map3_row(top3, correct))
q10 = round(float(np.mean(scores)), 4)
print(f'Q10: {q10}')

Q10: 1.0
